# Challenge — Module 2.3: Monthly Sales Report

**Format:** one report. You must use every concept from 2.3: filtering, `.groupby().agg()` with named aggregations, `pd.merge`, reshaping (`.pivot_table`), date handling (`pd.to_datetime` + extraction), and vectorized operations (no `.apply` allowed). Plus everything from 2.1–2.2.

**Scenario.** A small e-commerce shop wants a monthly report broken down by product category. You're given two CSVs: `orders` and `products`. Join, aggregate, reshape, and deliver a wide-format report.

## The problem

Setup writes two files:

- `orders.csv`: `order_id, product_id, qty, unit_price, order_ts` (ts is ISO string)
- `products.csv`: `product_id, name, category`

### Your task

Produce a single DataFrame `report` with the following shape:

- **Index:** `year_month` (e.g. `'2026-03'`) — strings, sorted ascending.
- **Columns:** one per category present in the data.
- **Values:** total revenue (`qty * unit_price`) for that category in that month, rounded to 2 decimals. Missing combinations -> `0.0`.

Then print:

1. The report.
2. The single `(year_month, category)` pair with the highest revenue.
3. Total revenue across **all** months for orders placed on a **weekday** (Mon–Fri).

### Expected output (shape)

```
report:
category    books  electronics  toys
year_month                          
2026-03      45.00       120.00  0.00
2026-04      30.00        80.00  60.00
...

top: ('2026-03', 'electronics') = 120.00
weekday_revenue=335.00
```

(Real numbers come from the setup data.)

### Restrictions

- Join `orders` with `products` using `pd.merge`, left join on `product_id`.
- Compute `revenue` as a vectorized column (`df['qty'] * df['unit_price']`), not via `.apply`.
- Extract `year_month` from `order_ts` using `pd.to_datetime` + `.dt.strftime('%Y-%m')`.
- Use `.pivot_table(..., fill_value=0)` for the wide format, **not** `.unstack` chains.
- For the top cell: `.stack().idxmax()` on the report works nicely.

### Hints

- `pd.to_datetime(df['order_ts']).dt.dayofweek` returns 0=Mon ... 6=Sun — weekdays are `< 5`.
- Build the long-form `(year_month, category, revenue)` first via `groupby([...]).agg(revenue=('revenue', 'sum'))`, then pivot.

In [ ]:
# --- setup: writes orders.csv and products.csv ---
import csv

products = [
    ['product_id', 'name',       'category'],
    ['P01',        'USB Cable',  'electronics'],
    ['P02',        'HDMI Cable', 'electronics'],
    ['P03',        'Notebook',   'books'],
    ['P04',        'Pen Set',    'books'],
    ['P05',        'Lego Brick', 'toys'],
]
orders = [
    ['order_id', 'product_id', 'qty', 'unit_price', 'order_ts'],
    ['O001', 'P01', '4',  '15.00', '2026-03-02T10:15:00'],  # Mon
    ['O002', 'P02', '2',  '30.00', '2026-03-05T14:00:00'],  # Thu
    ['O003', 'P03', '3',  '10.00', '2026-03-07T09:00:00'],  # Sat
    ['O004', 'P04', '5',  '3.00',  '2026-03-15T11:30:00'],  # Sun
    ['O005', 'P01', '1',  '15.00', '2026-04-01T16:45:00'],  # Wed
    ['O006', 'P02', '2',  '32.50', '2026-04-10T08:00:00'],  # Fri
    ['O007', 'P03', '6',  '5.00',  '2026-04-12T20:00:00'],  # Sun
    ['O008', 'P05', '4',  '15.00', '2026-04-20T13:00:00'],  # Mon
    ['O009', 'P05', '2',  '15.00', '2026-05-01T09:00:00'],  # Fri
    ['O010', 'P04', '10', '3.00',  '2026-05-09T17:00:00'],  # Sat
]
with open('products.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(products)
with open('orders.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(orders)
print('setup done')

setup done


In [ ]:
# your code here
import pandas as pd

orders   = pd.read_csv('orders.csv')
products = pd.read_csv('products.csv')

# 1. merge
# 2. vectorized revenue + year_month + is_weekday
# 3. groupby(year_month, category).agg(revenue=...)
# 4. pivot_table -> report
# 5. print report, top cell, weekday revenue
